# Criptografía básica: OTP, RSA y Diffie-Hellman

Implementación de los esquemas criptográficos fundamentales con énfasis en
su conexión con la teoría de la complejidad.

**Artículo relacionado:** `05/02-criptografia-y-complejidad.md`

In [ ]:
import math
import random

## 1. One-Time Pad (OTP): seguridad perfecta de Shannon

In [ ]:
def otp_encrypt(message: bytes, key: bytes) -> bytes:
    """Cifra con OTP: c = m XOR k."""
    assert len(key) >= len(message), "La clave debe ser al menos tan larga como el mensaje"
    return bytes(m ^ k for m, k in zip(message, key))


def otp_decrypt(ciphertext: bytes, key: bytes) -> bytes:
    """Descifra con OTP: m = c XOR k."""
    return otp_encrypt(ciphertext, key)  # XOR es su propia inversa


# Ejemplo
message = b'HOLA'
key = bytes([random.randint(0, 255) for _ in range(len(message))])
ciphertext = otp_encrypt(message, key)
decrypted = otp_decrypt(ciphertext, key)

print("One-Time Pad:")
print(f"  Mensaje:    {list(message)} = {message}")
print(f"  Clave:      {list(key)}")
print(f"  Cifrado:    {list(ciphertext)}")
print(f"  Descifrado: {decrypted}")
print(f"  Correcto:   {decrypted == message}")
print()

# Demostrar seguridad perfecta: cualquier plaintext es posible dado el ciphertext
print("Seguridad perfecta: para el cifrado", list(ciphertext))
print("el mensaje podría haber sido CUALQUIER cadena de 4 bytes:")
for possible_msg in [b'HOLA', b'ADIO', b'XXXX', b'\x00\x00\x00\x00']:
    implied_key = bytes(c ^ m for c, m in zip(ciphertext, possible_msg))
    print(f"  Si m={possible_msg}, entonces k={list(implied_key)} (igualmente probable)")

### Ejercicio 1.1: Ataque Two-Time Pad
Si Alicia reutiliza la misma clave OTP para dos mensajes, Eve puede obtener $m_1 \oplus m_2$.

In [ ]:
# Mensajes con formato conocido
m1 = b'ATACAR NORTE '
m2 = b'RETIRAR SUR  '
key_reused = bytes([random.randint(0, 255) for _ in range(max(len(m1), len(m2)))])

c1 = otp_encrypt(m1, key_reused[:len(m1)])
c2 = otp_encrypt(m2, key_reused[:len(m2)])

# Eve intercepta c1 y c2:
xor_ciphers = bytes(a ^ b for a, b in zip(c1, c2))
# Eve sabe que xor_ciphers = m1 XOR m2

# Si Eve adivina parte de m1 (por ejemplo, que empieza con 'ATACAR'):
known_prefix = b'ATACAR'
recovered_m2_prefix = bytes(xor_ciphers[i] ^ known_prefix[i] for i in range(len(known_prefix)))

print("Ataque two-time pad:")
print(f"  c1 XOR c2 = m1 XOR m2: {xor_ciphers}")
print(f"  Si sabemos que m1 empieza por '{known_prefix.decode()}':")
print(f"  m2 empieza por: {recovered_m2_prefix}")
print(f"  m2 real:        {m2[:len(known_prefix)]}")
print()
print("Lección: NUNCA reutilizar la clave OTP. La seguridad perfecta desaparece.")

# TODO para el estudiante:
# 1. ¿Qué información recupera Eve exactamente?
# 2. ¿Qué pasa si Eve conoce la estructura del formato (encabezados fijos)?
print("\nTODO: responde las preguntas del ejercicio en el enunciado")

## 2. Aritmética modular y RSA

In [ ]:
def mod_pow(base, exp, mod):
    """Exponenciación modular rápida: base^exp mod mod. O((log exp)²)."""
    return pow(base, exp, mod)  # Python lo hace eficientemente


def extended_gcd(a, b):
    """Algoritmo extendido de Euclides: devuelve (gcd, x, y) tal que ax + by = gcd."""
    if b == 0:
        return a, 1, 0
    g, x, y = extended_gcd(b, a % b)
    return g, y, x - (a // b) * y


def mod_inverse(e, phi):
    """Inverso modular de e en Z_phi: d tal que e*d ≡ 1 (mod phi)."""
    g, x, _ = extended_gcd(e, phi)
    if g != 1:
        raise ValueError(f"{e} no tiene inverso mod {phi}")
    return x % phi


def rsa_keygen(p, q, e=65537):
    """Genera claves RSA dados primos p, q y exponente público e."""
    n = p * q
    phi = (p - 1) * (q - 1)
    assert math.gcd(e, phi) == 1, f"e={e} no es coprimo con phi={phi}"
    d = mod_inverse(e, phi)
    return (e, n), (d, n), phi


def rsa_encrypt(m, public_key):
    """RSA: c = m^e mod n."""
    e, n = public_key
    assert 0 <= m < n, f"m={m} debe ser 0 ≤ m < n={n}"
    return mod_pow(m, e, n)


def rsa_decrypt(c, private_key):
    """RSA: m = c^d mod n."""
    d, n = private_key
    return mod_pow(c, d, n)


# Ejemplo del artículo: p=5, q=11, e=3
pub, priv, phi = rsa_keygen(p=5, q=11, e=3)
print(f"RSA con p=5, q=11:")
print(f"  n = {pub[1]}, phi(n) = {phi}")
print(f"  Clave pública: e={pub[0]}, n={pub[1]}")
print(f"  Clave privada: d={priv[0]}, n={priv[1]}")
print()

for m in [4, 7, 10, 1, 0]:
    c = rsa_encrypt(m, pub)
    m_dec = rsa_decrypt(c, priv)
    print(f"  m={m}: c={m}^{pub[0]} mod {pub[1]} = {c}, descifrado = {m_dec}, correcto = {m_dec == m}")

### Ejercicio 2.1: ¿Por qué RSA es seguro?
Sin conocer $p$ y $q$, ¿cómo calcularías $d$ a partir de $(e, n)$?

In [ ]:
# Ataque de factorización: si factorizas n, obtienes p y q, luego phi y d
def factor_brute(n):
    """Factoriza n por fuerza bruta: O(sqrt(n))."""
    for p in range(2, int(n**0.5) + 1):
        if n % p == 0:
            return p, n // p
    return n, 1  # primo


# Con n pequeño, la factorización es trivial
n_small = 5 * 11
p_found, q_found = factor_brute(n_small)
phi_found = (p_found - 1) * (q_found - 1)
d_found = mod_inverse(3, phi_found)

print(f"Ataque de factorización a n={n_small}:")
print(f"  Factores encontrados: p={p_found}, q={q_found}")
print(f"  phi(n) = {phi_found}")
print(f"  d = e^{{-1}} mod phi = {d_found}")
print(f"  ¿Coincide con la clave privada real ({priv[0]})? {d_found == priv[0]}")
print()

# Con n grande, la factorización es inviable
import time
n_medium = 104723 * 104729  # dos primos medianos
t0 = time.time()
p_m, q_m = factor_brute(n_medium)
t_factor = time.time() - t0
print(f"Factorizar n={n_medium} (2 primos ~10^5): {t_factor:.4f} segundos")
print(f"Para n de 2048 bits (~10^616): imposible con fuerza bruta")
print()

# TODO para el estudiante:
# 1. ¿Por qué el algoritmo de Shor factoriza en tiempo polinomial en un ordenador cuántico?
# 2. ¿Qué tamaño de n es seguro hoy (2024) contra ataques clásicos?
print("TODO: investiga el algoritmo de Shor y responde las preguntas")

## 3. Diffie-Hellman: intercambio de claves

In [ ]:
def diffie_hellman(p, g):
    """
    Protocolo Diffie-Hellman sobre Z_p con generador g.
    
    Alice y Bob establecen un secreto compartido sin transmitirlo.
    """
    # Alicia elige a en secreto
    a = random.randint(2, p - 2)
    # Bob elige b en secreto
    b = random.randint(2, p - 2)
    
    # Intercambio público
    A = mod_pow(g, a, p)  # Alicia envía A = g^a mod p
    B = mod_pow(g, b, p)  # Bob envía B = g^b mod p
    
    # Secreto compartido
    shared_alice = mod_pow(B, a, p)  # Alice calcula B^a = g^{ba}
    shared_bob   = mod_pow(A, b, p)  # Bob calcula A^b = g^{ab}
    
    assert shared_alice == shared_bob, "Error: secretos no coinciden"
    
    return {
        'p': p, 'g': g,
        'a_privado': a, 'b_privado': b,
        'A_publico': A, 'B_publico': B,
        'secreto_compartido': shared_alice
    }


# Primo de 8 bits para demostración (en práctica: 2048+ bits)
p_dh = 233  # primo
g_dh = 3    # generador primitivo de Z_233

random.seed(42)
resultado = diffie_hellman(p_dh, g_dh)

print("Protocolo Diffie-Hellman:")
print(f"  Parámetros públicos: p={resultado['p']}, g={resultado['g']}")
print(f"  Alicia envía A = g^a mod p = {resultado['A_publico']} (a={resultado['a_privado']} en secreto)")
print(f"  Bob envía   B = g^b mod p = {resultado['B_publico']} (b={resultado['b_privado']} en secreto)")
print(f"  Secreto compartido = g^{{ab}} mod p = {resultado['secreto_compartido']}")
print()
print("Eve ve: p, g, A, B — pero no puede calcular g^{ab} sin saber a o b.")
print("Esto requiere el problema del logaritmo discreto: dado g^a mod p, calcular a.")
print()

# Logaritmo discreto por fuerza bruta (solo funciona para p pequeño)
def discrete_log_brute(g, A, p):
    """Resuelve g^x ≡ A (mod p) por fuerza bruta."""
    for x in range(1, p):
        if mod_pow(g, x, p) == A:
            return x
    return None

a_recovered = discrete_log_brute(g_dh, resultado['A_publico'], p_dh)
print(f"Ataque por fuerza bruta: a recuperado = {a_recovered} (real: {resultado['a_privado']})")
print(f"Para p de 2048 bits, este ataque requeriría 2^2048 operaciones — imposible.")

In [ ]:
# === Tests automáticos ===

# Test 1: OTP — cifrado y descifrado son inversos
msg = b'hola'
key = b'\xab\xcd\xef\x12'
ciphertext = otp_encrypt(msg, key)
plaintext = otp_decrypt(ciphertext, key)
assert plaintext == msg, f'OTP roundtrip falló: {plaintext!r} != {msg!r}'

# Test 2: OTP — el cifrado es XOR puro
assert ciphertext == bytes(a ^ b for a, b in zip(msg, key))

# Test 3: mod_pow — exponenciación modular
assert mod_pow(2, 10, 1000) == pow(2, 10, 1000)  # = 24
assert mod_pow(3, 100, 97) == pow(3, 100, 97)     # pequeño primo
assert mod_pow(2, 0, 7) == 1                       # x^0 = 1

# Test 4: RSA — cifrar y descifrar recupera el mensaje
# Usamos primos pequeños para el test
p_test, q_test = 61, 53
n_test = p_test * q_test  # = 3233
e_test = 17
phi_test = (p_test - 1) * (q_test - 1)  # = 3120
_, _, d_test = extended_gcd(e_test, phi_test)
d_test = d_test % phi_test
m_test = 65  # mensaje de prueba
c_test = mod_pow(m_test, e_test, n_test)
m_recovered = mod_pow(c_test, d_test, n_test)
assert m_recovered == m_test, f'RSA roundtrip: esperado {m_test}, obtenido {m_recovered}'

# Test 5: Diffie-Hellman — Alice y Bob calculan el mismo secreto
# Usamos p=23, g=5 (ejemplo estándar)
p_dh, g_dh = 23, 5
result = diffie_hellman(p_dh, g_dh)
# La función debe retornar un dict con 'shared_secret' o similar
# o simplemente verificar que Alice y Bob coinciden
if isinstance(result, dict):
    alice_secret = result.get('alice_shared') or result.get('shared_secret')
    bob_secret = result.get('bob_shared') or result.get('shared_secret')
    if alice_secret is not None and bob_secret is not None:
        assert alice_secret == bob_secret, f'DH: Alice={alice_secret} != Bob={bob_secret}'

print('✓ Todos los tests de criptografia-basica pasaron correctamente.')